# M1-B2 — Classer les avis par motif d'insatisfaction

**Mode d'emploi.** Exécute les cellules dans l'ordre. Les `___` sont à compléter. Après chaque étape, le bloc
✅ indique le résultat que tu dois obtenir (à quelques centièmes près) : s'il est très différent, relis l'étape
avant d'avancer. Les cellules sans `___` sont fournies : lis-les, elles font partie de l'apprentissage.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"   # avant tout « import keras » : Keras utilisera PyTorch

import json
import time
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import LabelEncoder

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
CACHE = DATA / "cache"
CACHE.mkdir(exist_ok=True)
MODELS = ROOT / "models"

MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## Étape 1 — Démarrer

Cette cellule télécharge le modèle d'embeddings au premier lancement (environ 470 Mo, quelques minutes) :
lance-la tout de suite, et continue à lire pendant ce temps.

In [ ]:
from sentence_transformers import SentenceTransformer

encoder = SentenceTransformer(MODEL_NAME)
encoder.encode(["Le lit était très inconfortable.", "Le matelas était défoncé."]).shape

✅ `(2, 384)` : deux phrases, 384 nombres chacune.

## Étape 2 — Explorer le corpus

- `avis_motifs.csv` : 1 200 avis de 2016, étiquetés par un LLM → pour **entraîner et tester**.
- `avis_recents.csv` : 180 avis de 2017, étiquetés à la main → **uniquement pour évaluer**, à la fin.

In [ ]:
avis = pd.read_csv(DATA / "avis_motifs.csv")
recents = pd.read_csv(DATA / "avis_recents.csv")
print(len(avis), "avis,", len(recents), "avis récents")
avis["motif"].value_counts()

✅ 9 motifs, de `proprete` (189 avis) à `emplacement` (67). Les classes sont **déséquilibrées** : c'est pourquoi on mesurera le F1 **macro** (chaque motif compte autant).

In [ ]:
avis["texte"].str.len().describe().round()

In [ ]:
pd.set_option("display.max_colwidth", 200)
avis.sample(20, random_state=0)[["avis_id", "texte", "motif"]]

### ✏️ Ton avis sur les étiquettes du LLM

Lis les 20 avis ci-dessus. Pour combien es-tu en désaccord avec l'étiquette ? Note ici 2 ou 3 `avis_id` douteux et pourquoi.

*…*

## Étape 3 — La référence : TF-IDF + régression logistique

On découpe en 75 % entraînement / 25 % test, **stratifié** (chaque motif garde sa proportion dans les deux jeux).

In [ ]:
train, test = train_test_split(avis, test_size=0.25, stratify=___, random_state=42)

def f1_macro(y_true, y_pred):
    return round(f1_score(y_true, y_pred, average="macro"), 3)

tfidf = make_pipeline(TfidfVectorizer(), LogisticRegression(max_iter=2000))
tfidf.fit(___, ___)
print("TF-IDF — F1 macro test :", f1_macro(test["motif"], tfidf.predict(___)))

✅ Environ **0,91**. C'est la référence à battre.

## Étape 4 — Les embeddings

👉 Mini-cours 11. On encode chaque avis **une seule fois** et on garde les vecteurs en cache : la fonction est fournie.

In [ ]:
def encode_cached(textes, nom):
    chemin = CACHE / f"{nom}.npy"
    if chemin.exists():
        return np.load(chemin)
    vecteurs = encoder.encode(list(textes), batch_size=64, show_progress_bar=True)
    np.save(chemin, vecteurs)
    return vecteurs

E_avis = encode_cached(avis["texte"], "avis")          # une ligne par avis, dans l'ordre de `avis`
E_recents = encode_cached(recents["texte"], "recents")
E_avis.shape, E_recents.shape

✅ `(1200, 384)` et `(180, 384)`. Le premier lancement prend de quelques secondes à une minute ; le suivant est instantané.

⚠️ **Le piège de l'alignement.** `train` et `test` sont mélangés : il faut prendre les vecteurs **dans le même ordre**.
`avis.index` va de 0 à 1 199 dans l'ordre du fichier, comme les lignes de `E_avis` : on peut donc indexer par `train.index`.

In [ ]:
E_train = E_avis[train.index]
E_test = E_avis[___]

emb_lr = LogisticRegression(max_iter=3000).fit(___, train["motif"])
print("Embeddings + logreg — F1 macro test :", f1_macro(test["motif"], emb_lr.predict(E_test)))

✅ Environ **0,86** : un peu moins bien que TF-IDF sur ce test. Garde ce résultat en tête pour l'étape 6.

In [ ]:
pd.DataFrame({
    "tfidf": f1_score(test["motif"], tfidf.predict(test["texte"]), average=None),
    "embeddings": f1_score(test["motif"], emb_lr.predict(E_test), average=None),
}, index=sorted(avis["motif"].unique())).round(2)

## Étape 5 — Un petit réseau Keras

👉 Mini-cours 12. Keras attend des classes **numérotées** : `LabelEncoder` fait la conversion.

In [ ]:
import keras

keras.utils.set_random_seed(42)           # pour des résultats reproductibles
le = LabelEncoder().fit(avis["motif"])
y_train = le.transform(train["motif"])

reseau = keras.Sequential([
    keras.Input(shape=(384,)),
    keras.layers.Dense(___, activation="relu"),        # nombre de neurones de la couche cachée : 64 à 256
    keras.layers.Dropout(0.3),
    keras.layers.Dense(___, activation="softmax"),   # une sortie par motif
])
reseau.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
reseau.summary()

In [ ]:
arret = keras.callbacks.EarlyStopping(patience=___, restore_best_weights=True)

historique = reseau.fit(
    E_train, y_train,
    validation_split=0.2,      # train est déjà mélangé : on peut prendre ses 20 % de fin
    epochs=100, batch_size=32, verbose=0,
    callbacks=[___],
)
print("arrêt après", len(historique.history["loss"]), "époques")

In [ ]:
plt.plot(historique.history["loss"], label="entraînement")
plt.plot(historique.history["val_loss"], label="validation")
plt.xlabel("époque"); plt.ylabel("perte (log loss)"); plt.legend(); plt.show()

✅ Arrêt vers la 20e époque. La perte d'entraînement baisse toujours ; celle de validation se stabilise.

### ✏️ Que montrent tes courbes ? Pourquoi l'arrêt précoce s'est-il déclenché ?

*…*

In [ ]:
def predict_reseau(E):
    return le.inverse_transform(reseau.predict(E, verbose=0).argmax(axis=1))

print("Keras — F1 macro test :", f1_macro(test["motif"], predict_reseau(E_test)))

✅ Environ **0,85**.

## Étape 6 — Comparer et conclure

Jusqu'ici, tout a été mesuré sur le **test**, tiré du même corpus que l'entraînement.
Les **avis récents**, eux, ont été écrits par d'autres clients, avec d'autres mots. C'est le moment de vérité.

In [ ]:
comparaison = pd.DataFrame([
    {"modèle": "TF-IDF + logreg",
     "F1 test": f1_macro(test["motif"], tfidf.predict(test["texte"])),
     "F1 récents": f1_macro(recents["motif"], tfidf.predict(___))},
    {"modèle": "embeddings + logreg",
     "F1 test": f1_macro(test["motif"], emb_lr.predict(E_test)),
     "F1 récents": f1_macro(recents["motif"], emb_lr.predict(___))},
    {"modèle": "embeddings + Keras",
     "F1 test": f1_macro(test["motif"], predict_reseau(E_test)),
     "F1 récents": f1_macro(recents["motif"], predict_reseau(___))},
])
comparaison

### ✏️ Le classement des modèles est-il le même sur les deux jeux ? Pourquoi, à ton avis ?

*Pense aux limites de TF-IDF vues en cours.*

In [ ]:
ConfusionMatrixDisplay.from_predictions(recents["motif"], predict_reseau(E_recents), xticks_rotation=45)
plt.show()

### Examiner 10 erreurs

La cellule affiche 10 avis du test où le réseau ne prédit pas l'étiquette du LLM.

In [ ]:
erreurs = test.assign(prediction=predict_reseau(E_test)).query("motif != prediction")
print(len(erreurs), "erreurs sur", len(test))
erreurs[["avis_id", "texte", "motif", "prediction"]].head(10)

### ✏️ Pour chaque erreur : le modèle se trompe-t-il, ou l'étiquette du LLM est-elle douteuse ?

| avis_id | étiquette (LLM) | prédiction | qui a raison, selon toi ? |
|---|---|---|---|
| | | | |

## Sauvegarder le modèle retenu

In [ ]:
choix = reseau            # ou emb_lr : à toi de choisir, et de le justifier dans la synthèse

MODELS.mkdir(exist_ok=True)
reseau.save(MODELS / "motifs.keras")
(MODELS / "motifs_info.json").write_text(json.dumps({
    "model_name": "motifs",
    "trained_on": date.today().isoformat(),
    "embedding_model": MODEL_NAME,
    "classes": list(___),   # l'ordre des sorties du réseau : indispensable pour s'en servir
}, indent=2, ensure_ascii=False), encoding="utf-8")
sorted(p.name for p in MODELS.iterdir())

## ✏️ Synthèse (10 lignes)

- Démarche :
- Comparaison (test et avis récents) :
- Modèle retenu et pourquoi :
- Ce que tu as appris des erreurs :
- Limites :